# Lab 01 — SFT From Scratch (★★☆ · CORE)

Prototype workspace. Full assignment & contract details in `README.md`;
implementable stubs in `starter.py`; shared config in `configs/01_sft_from_scratch.yaml`.

**Prereqs done?** Lab 00 · [Lecture 2 pt.1](https://www.youtube.com/watch?v=4gIwiSPmQkU&list=PLL1tdVxB1CpVpEtMHxwuR4uI4Lxjw00_y&index=3) · [Ch. 4](https://rlhfbook.com/c/04-instruction-tuning)

> TODO(workflow): prototype each function here first (numpy/torch free-form), then paste the
> implementation into `starter.py`, keeping `tests/` green with `python3 -m pytest tests/ -q`.
> Export the finished loop as a config-driven `train.py` for GT-cluster tmux runs.

## 0. Setup

- Colab T4: pinned deps + `HF_TOKEN` secret (see LABS_SETUP.md); fp16 instead of bf16 on T4.
- Cluster A40: venv at `/coc/scratch/vchopra/venvs_post_training/`; outputs to `/coc/scratch/vchopra/post_training_labs/01_sft_from_scratch/`.
- No HF `Trainer`, no TRL — everything from scratch.

In [ ]:
# TODO(00.setup): pip installs, imports, load configs/01_sft_from_scratch.yaml, seed_everything

## 1. Dataset → prompt-masked rows (`build_prompt_masked_labels`)

Contract: template twice (prompt-only w/ generation prompt vs full), supervise exactly
[prompt_end … EOS], truncate before masking, skip rows without a trainable assistant tail.

**Checkpoint:** print one conversation as `(token, id, label)` triples and explain every row.

In [ ]:
# TODO(01.data): implement build_prompt_masked_labels here; visualize masks for 3 rows

## 2. Collate to rectangular batches (`collate_sft`, `SFTBatch`)

Right-pad ids with `pad_token_id`, attention_mask 0 at pads, labels `IGNORE_INDEX(-100)` at pads.
`tests/test_01_sft_from_scratch.py::test_collate_sft_contract` un-skips once implemented.

In [ ]:
# TODO(02.collate): implement collate_sft; run make_dataloader over no_robots subset; batch smoke-check

## 3. Masked causal-LM loss (`compute_loss`)

Forward once; shift logits[:, :-1] vs labels[:, 1:]; CE with `ignore_index=-100`.
Sanity check: batch of ALL-masked labels must not silently produce a real loss value you trust.
Then recompute by hand over unmasked positions only in numpy — the two numbers must match.

In [ ]:
# TODO(03.loss): compute_loss + hand-rolled mean-over-unmasked cross-check on a fake logits tensor

## 4. From-scratch checkpoint — LoRA layer

Frozen base `W` + trainable `A:(r,in)`, `B:(out,r)`, forward = base(x) + α/r·B(A(x)).
Init: A ~ Gaussian, B = zeros (why not both zero?).

**Verification gate:** frozen-linear + adapter forward vs your layer on identical random input.
Then SFT Qwen3-0.6B with it on Colab (`r ∈ {8,32}` vs full-FT).

In [ ]:
# TODO(04.lora): LoRALinear.__init__/forward/merge + apply_lora; equivalence check vs wrapped baseline

## 5. Training loop (`train_step`, `run_training_loop`)

bf16 autocast · grad accumulation ÷ steps · clip-grad-norm · warmup→decay scheduler ·
JSONL metrics per optimizer step · periodic val loss · step-0/final generation panels.

**Metrics that must move:** train/val loss ↓, grad norm sane, response length changes,
panel quality shifts.

In [ ]:
# TODO(05.loop): train_step + run_training_loop; small smoke run (500 rows, 1 epoch, lr 5e-6)

## 6. Generation panels + experiments (`generation_panel`)

Fixed 6-prompt panel, greedy AND sampled, step-0 vs final side-by-side.
Experiments grid (README §7): lr × epochs × data size × {LoRA r8, r32, full-FT}.
Optional: naive packing via `pack_sequences` — throughput gain vs contamination risk write-up.

In [ ]:
# TODO(06.panels): generation_panel; run experiment sweep; collect metrics.jsonl artifacts back here

## 7. Analysis — completion criteria (README §10)

- [ ] base-vs-SFT transcripts on held-out prompts
- [ ] narrated loss plot (annotate early cliff / instability)
- [ ] LoRA-vs-full-FT comparison table
- [ ] Answers to README §8 questions written inline below this cell

Only after all four boxes ticked: open the answer key
`rlhf-book/code/instruction_tuning/{train.py,utils.py}` and diff decisions into README §11.